In [1]:
from pathlib import Path
import pandas as pd


def find_project_root():
    """Find the workspace directory containing the training and output paths."""
    for directory in (Path.cwd(), *Path.cwd().parents):
        if (directory / 'dataset' / 'train' / 'train_ground_truth.tsv').is_file():
            return directory
    raise FileNotFoundError(
        'Could not find project root containing dataset/train/train_ground_truth.tsv.'
    )


def evaluate_predictions():
    project_root = find_project_root()
    gt_path = project_root / 'dataset' / 'train' / 'train_ground_truth.tsv'
    pred_path = project_root / 'output' / 'matching_results.tsv'

    if not gt_path.is_file() or not pred_path.is_file():
        print(
            'Error: Ground truth or prediction file not found. '
            f'\nGround truth: {gt_path}'
            f'\nPredictions: {pred_path}'
        )
        return

    gt_df = pd.read_csv(gt_path, sep='\t').fillna('')
    pred_df = pd.read_csv(pred_path, sep='\t').fillna('')

    gt_dict = gt_df.set_index('source1_entity_id')['matched_entity_ids'].apply(
        lambda x: set(m.strip() for m in str(x).split(',') if m.strip())
    ).to_dict()

    pred_dict = pred_df.set_index('source1_entity_id')['matched_entity_ids'].apply(
        lambda x: set(m.strip() for m in str(x).split(',') if m.strip())
    ).to_dict()

    target_ids = [
    "S1-559551054",
    "S1-580254632",
    "S1-405795474",
    "S1-765456522",
    "S1-55333937",
    "S1-876256279",
    "S1-651095976",
    "S1-522840559",
    "S1-601945198",
    "S1-757082720",
    "S1-590134263",
    "S1-842497118",
    "S1-326704888"
]

    print('\n' + '=' * 70)
    print(f"{'S1 ID':<15} | {'Matched (TP)':<12} | {'Extra (FP)':<12} | {'Missing (FN)':<12}")
    print('=' * 70)

    total_tp, total_pred, total_actual = 0, 0, 0

    for tid in target_ids:
        preds = pred_dict.get(tid, set())
        actuals = gt_dict.get(tid, set())

        tp = len(preds.intersection(actuals))
        fp = len(preds - actuals)
        fn = len(actuals - preds)

        total_tp += tp
        total_pred += len(preds)
        total_actual += len(actuals)

        print(f"{tid:<15} | {tp:<12} | {fp:<12} | {fn:<12}")

    precision = total_tp / total_pred if total_pred > 0 else 0.0
    recall = total_tp / total_actual if total_actual > 0 else 0.0
    beta = 0.5
    f05 = (
        ((1 + beta**2) * precision * recall) /
        ((beta**2 * precision) + recall)
        if (precision + recall) > 0 else 0.0
    )

    print('=' * 70)
    print('EVALUATION SUMMARY:')
    print(f'Total True Positives (Matched) : {total_tp}')
    print(f'Total Model Predictions        : {total_pred}')
    print(f'Total Actual Ground Truth      : {total_actual}')
    print(f'Precision                      : {precision:.4f}')
    print(f'Recall                         : {recall:.4f}')
    print(f'F_0.5 Score                    : {f05:.4f}')
    print('=' * 70)


evaluate_predictions()


S1 ID           | Matched (TP) | Extra (FP)   | Missing (FN)
S1-559551054    | 4            | 0            | 0           
S1-580254632    | 3            | 0            | 0           
S1-405795474    | 3            | 0            | 0           
S1-765456522    | 3            | 0            | 0           
S1-55333937     | 0            | 1            | 0           
S1-876256279    | 2            | 0            | 0           
S1-651095976    | 3            | 0            | 0           
S1-522840559    | 6            | 0            | 0           
S1-601945198    | 2            | 0            | 0           
S1-757082720    | 3            | 0            | 0           
S1-590134263    | 4            | 0            | 0           
S1-842497118    | 3            | 0            | 0           
S1-326704888    | 4            | 0            | 1           
EVALUATION SUMMARY:
Total True Positives (Matched) : 40
Total Model Predictions        : 41
Total Actual Ground Truth      : 41
Precision        

In [7]:
import pandas as pd


def scan_dataset_languages():
    project_root = find_project_root()
    source_path = project_root / 'dataset' / 'train' / 'train_source2.tsv'
    s1 = pd.read_csv(
        source_path,
        sep='\t',
        usecols=['entity_id', 'business_name', 'country'],
        dtype=str,
        keep_default_na=False,
    )

    non_ascii = s1['business_name'].str.contains(r'[^\x00-\x7F]', regex=True, na=False)
    foreign_samples = s1.loc[non_ascii].head(10)

    print(f"Total rows scanned: {len(s1)}")
    print(f"Rows with foreign/accented characters: {int(non_ascii.sum())}")
    print("\n--- Sample Foreign/Accented Business Names ---")
    for row in foreign_samples.itertuples(index=False):
        print(f"ID: {row.entity_id} | Country: {row.country} | Name: {row.business_name}")


scan_dataset_languages()

Total rows scanned: 5034616
Rows with foreign/accented characters: 764608

--- Sample Foreign/Accented Business Names ---
ID: S2-166376419 | Country: India | Name: राम मार्केटिंग प्राइवेट लिमिटेड
ID: S2-639257739 | Country: India | Name: आदित्य प्रॉपर्टीज एलएलपी
ID: S2-615967906 | Country: India | Name: सन कंस्ट्रक्शंस प्राइवेट लिमिटेड
ID: S2-566688803 | Country: India | Name: रियल मॉडर्न फूड लिमिटेड
ID: S2-879667609 | Country: India | Name: குளோபல் பிசினஸ் பிரைவேட் லிமிடெட்
ID: S2-783158461 | Country: US | Name: Animal Welfare Nétwork
ID: S2-978126813 | Country: India | Name: Private Ambernath Sólar Limited
ID: S2-348280942 | Country: India | Name: શક્તિ અર્બન પ્રોડક્ટ્સ પ્રાઇવેટ લિમિટેડ
ID: S2-64510343 | Country: India | Name: अल्फा अल टेक्नोलॉजीज प्राइवेट लिमिटेड
ID: S2-631299191 | Country: India | Name: ग्लोबल इन्वेस्टमेंट प्रा. लि.


In [11]:
from pathlib import Path
import pickle
import sys
import pandas as pd


def compare_saved_models():
    project_root = find_project_root()
    src_path = str(project_root / 'src')
    if src_path not in sys.path:
        sys.path.insert(0, src_path)

    model_paths = {
        'Neural network': project_root / 'model' / 'nn_model.pkl',
        'Random forest': project_root / 'model' / 'randomforest_model.pkl',
    }

    rows = []
    for model_name, model_path in model_paths.items():
        if not model_path.is_file():
            print(f'{model_name}: missing {model_path}')
            print('Run src/nn_train.py and src/run_lightweight_pipeline.py first.')
            continue

        with model_path.open('rb') as model_file:
            bundle = pickle.load(model_file)

        estimator = bundle['model']
        rows.append({
            'model': model_name,
            'class': type(estimator).__name__,
            'features': ', '.join(bundle.get('feature_columns', [])),
            'architecture': bundle.get('architecture', 'not recorded'),
            'threshold': bundle.get('threshold', 'n/a'),
            'classes': str(getattr(estimator, 'classes_', 'not applicable')),
            'label_counts': str(bundle.get('training_label_counts', {})),
            'file': str(model_path.relative_to(project_root)),
        })

    if rows:
        comparison = pd.DataFrame(rows)
        print('\nSaved model comparison')
        print('=' * 110)
        print(comparison.to_string(index=False))
        print('=' * 110)


compare_saved_models()

Random forest: missing c:\DevelopmentWorks\amazondataset\student_resource\model\randomforest_model.pkl
Run src/nn_train.py and src/run_lightweight_pipeline.py first.

Saved model comparison
         model         class                                   features architecture  threshold classes         label_counts               file
Neural network MLPClassifier name_max_jaro, address_jaro, country_match not recorded        0.5   [0 1] {0: 690169, 1: 3445} model\nn_model.pkl
